# 🪞 A Reflective Dynamic Planning Agent for Multi-Step Queries

## Learning Objectives
In this notebook, you will learn:
1. **A nested research agent** - A ReAct agent with a web-extraction tool and a context budget, used as one component
2. **Planning with a structured plan** - A planner chain that turns a question into at most five steps
3. **Replanning after every step** - A replanner that reflects on the results so far and either revises the plan or answers
4. **A hierarchical plan-execute-replan graph** - Planner → nested ReAct executor → replanner, with a step cap enforced in code
5. **Proving the cap without an LLM** - Stub components that never finish, to show the graph still stops

## Where this fits
`02_planning_decompose_execute_replan.ipynb` replanned only when a per-step evaluator said a step failed, and every step was a single search. Here each step is delegated to a **whole ReAct agent** (which decides its own searches), and the replanner reviews the plan after **every** step - deciding whether to keep going, change the remaining steps, or answer. That nesting makes this a **hierarchical compound agent**: one system whose executor is itself an agent, not a team of independent agents. This is the last notebook of the planning module; module 11 turns the "look back at your own work" idea into its own pattern - reflection.

## Prerequisites
- The ReAct loop in LangGraph - `modules/03_react_agents/notebooks/01_react_agent_langgraph.ipynb`
- `02_planning_decompose_execute_replan.ipynb` (replanning and stopping rules)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` and `TAVILY_API_KEY` in the project-root `.env`

---
## 📖 Part 1: The Reflective Dynamic Planning Agent

This agent generates a dynamic plan and iterates on it, using reflection to help with multi-step complex query analysis.

![](https://i.imgur.com/odnq3de.png)

This project focuses on building a **Reflective Dynamic Planning Agent for Multi-Step Complex Query Analysis**, designed to break down user queries or tasks into manageable steps, execute them iteratively, and update the plan dynamically through reflection and replanning. The components of the system are as follows:

1. **Plan Creation**:
   - The agent begins by analyzing the user's input (topic or question) to create a **step-by-step plan** of sub-questions or steps needed to solve the overall task.
   - Each step is clearly outlined in the **step-by-step plan** for sequential execution.

2. **Step Execution**:
   - The agent executes one step at a time using a nested **ReAct research agent** (`gpt-4o-mini`).
   - During execution:
     - External tools, such as the **web search tool**, are used to gather required information or data for the step.
     - The result of the executed step is captured and stored in the agent's state.

3. **Reflection and Replanning**:
   - After each step, the system evaluates the results and sends the **overall plan**, **steps executed**, **results**, and **remaining steps** to the **Replanner**.
   - The **Replanner**:
     - Assesses whether the remaining steps need modification based on the current results and any new insights.
     - Updates the remaining steps if necessary, ensuring alignment with the overall objective.
   - This iterative loop ensures adaptability to errors, incomplete data, or unforeseen circumstances.

4. **Dynamic Feedback Loop**:
   - The system monitors the status of the response:
     - If all steps are executed successfully or an error occurs, the process halts.
     - If there are remaining steps, the agent continues executing them one at a time, integrating results dynamically.

5. **Agent State Maintenance**:
   - Throughout the process, the agent maintains a record of:
     - The **original plan**.
     - **Executed steps and their results**.
     - **Remaining steps** for further execution.
   - This stateful approach ensures traceability and transparency in the agent’s decision-making process.

6. **Final Response Compilation**:
   - Once all steps are executed or the process encounters an error, the agent compiles everything to generate the final reponse.

| Property | Value |
|---|---|
| Origin | LangGraph [plan-and-execute tutorial](https://langchain-ai.github.io/langgraph/tutorials/plan-and-execute/plan-and-execute/), after Wang et al., *Plan-and-Solve Prompting* (2023) |

---
## 📦 Step 1: Environment Setup

Load the API keys and create the chat model through the course factory.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY and TAVILY_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: Research agent, planner and replanner
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🔎 Part 2: The Nested Research Agent

### 2.1 🧰 A Web-Extraction Tool with a Token Budget

The research agent's tool returns the **full page text** of the top search hits, not just snippets - which can easily be tens of thousands of tokens. `process_and_truncate_texts(...)` truncates a list of documents to a shared token budget, so each tool result stays within a fixed size no matter what the pages contain.

In [ ]:
# ============================================================================
# TOKEN BUDGET: Truncate a list of documents to a shared token limit
# ============================================================================
import tiktoken

def process_and_truncate_texts(text_docs, max_tokens=127000):
    """
    Calculate token counts for a list of text documents and truncate them to fit within max_tokens.

    Args:
        text_docs (list of str): List of text documents to process.
        max_tokens (int): Maximum token limit for truncation.

    Returns:
        list of str: Truncated list of text documents.
    """
    # Load the tokenizer (adjust the encoding name as per your use case, e.g., gpt-3.5-turbo, gpt-4, etc.)
    encoding = tiktoken.encoding_for_model("gpt-4o")

    # Calculate token counts for each document
    tokenized_docs = []
    for text in text_docs:
        tokens = encoding.encode(text)
        tokenized_docs.append(tokens)

    # Flatten all tokens into a single list
    all_tokens = [token for tokens in tokenized_docs for token in tokens]

    # Truncate to the maximum token limit
    truncated_tokens = all_tokens[:max_tokens]

    # Decode the truncated tokens back to text
    truncated_texts = []
    remaining_tokens = truncated_tokens[:]
    for tokens in tokenized_docs:
        if len(remaining_tokens) >= len(tokens):
            # Add the full document if its tokens fit within the remaining budget
            truncated_texts.append(encoding.decode(tokens))
            remaining_tokens = remaining_tokens[len(tokens):]
        else:
            # Partially add tokens of the last fitting document
            truncated_texts.append(encoding.decode(remaining_tokens))
            break

    return truncated_texts


# Try it on three tiny documents with a 20-token budget
text_docs = [
    "This is the first document. It has some content.",
    "The second document might have more content than the first.",
    "This is the third document. It also has its own content.",
]
print(process_and_truncate_texts(text_docs, max_tokens=20))

Only the first document and part of the second fit in 20 tokens - the third is dropped. The tool below uses the same function with a budget of `SEARCH_TOKEN_BUDGET` tokens per call (the original notebook allowed 127,000 tokens per call, which alone can exhaust a per-minute token limit).

In [ ]:
# ============================================================================
# TOOL: Search the web and return truncated page text
# ============================================================================
from langchain_core.tools import tool
from tavily import TavilyClient

tavily_client = TavilyClient()  # reads TAVILY_API_KEY
SEARCH_TOKEN_BUDGET = 4000      # total tokens of page text one tool call may return
search_log: list[str] = []      # every query actually sent


@tool
def search_web_extract_info(query: str) -> list:
    """Search the web for a query and extracts useful information from the search links."""
    print(f"   🔎 TOOL: searching for '{query}'")
    search_log.append(query)
    results = tavily_client.search(query=query, max_results=2, include_raw_content=True)
    docs = [doc.get("raw_content") or doc.get("content", "") for doc in results["results"]]
    return process_and_truncate_texts(docs, max_tokens=SEARCH_TOKEN_BUDGET)


docs = search_web_extract_info.invoke("what is langgraph")
print(f"✅ {len(docs)} documents, first one starts: {docs[0][:200]!r}")

### 2.2 🤖 The ReAct Research Agent

This agent (a *sub-agent*, or more precisely a nested agentic component) will be used by the planner to research each step of the plan. It is the standard ReAct loop from module 03, plus `trim_messages`: before each model call, the conversation is trimmed to the most recent messages that fit a token budget, starting on a human message so a tool result is never separated from the call that requested it.

In [ ]:
# ============================================================================
# REACT RESEARCH AGENT: chatbot <-> tools, with a context budget
# ============================================================================
from typing import Annotated

from langchain_core.messages import SystemMessage, trim_messages
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict


class State(TypedDict):
    messages: Annotated[list, add_messages]


tools = [search_web_extract_info]
llm_with_tools = llm.bind_tools(tools)

SYS_MSG = SystemMessage(content="Act as a helpful assistant and answer user questions")


def chatbot(state: State):
    messages = trim_messages(
        state["messages"],
        max_tokens=50_000,
        strategy="last",
        token_counter=llm,   # counts tokens with the model's own tokenizer
        start_on="human",
        allow_partial=False,
    )
    return {"messages": [llm_with_tools.invoke([SYS_MSG] + messages)]}


graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=tools))
graph_builder.add_edge(START, "chatbot")
graph_builder.add_conditional_edges("chatbot", tools_condition, ["tools", END])
# Any time a tool is called, we return to the chatbot to decide the next step
graph_builder.add_edge("tools", "chatbot")
react_agent = graph_builder.compile()

print("🧩 Research agent nodes:", [n for n in react_agent.get_graph().nodes if not n.startswith("__")])

In [ ]:
# ============================================================================
# TEST THE RESEARCH AGENT: One question, printed message by message
# ============================================================================
result = react_agent.invoke({"messages": [("user", "Who founded Nvidia, and in which year?")]})
for message in result["messages"]:
    content = message.content if isinstance(message.content, str) else str(message.content)
    calls = [c["args"] for c in getattr(message, "tool_calls", [])]
    print(f"[{message.type}] {calls or ''} {content[:300]}")

---
## 📋 Part 3: Planner and Replanner

### 3.1 🗺️ The Plan State and the Planner

`PlanExecute` carries the question, the current plan, the executed steps with their results (appended through an `operator.add` reducer), and the final response. The planner is a prompt + structured-output chain returning a `Plan` of at most five steps.

In [ ]:
# ============================================================================
# PLAN STATE + SCHEMA
# ============================================================================
import operator
from typing import List, Literal, Tuple, Union

from pydantic import BaseModel, Field


# state of the executed plan including overall plan, current status in terms of steps executed and output
# response is the final answer (or the reason the agent stopped)
class PlanExecute(TypedDict):
    input: str
    plan: List[str]
    past_steps: Annotated[List[Tuple], operator.add]
    response: str


class Plan(BaseModel):
    """Plan to follow in future to get to a well-researched output for user query"""

    type: Literal["Plan"]
    steps: List[str] = Field(description="Different steps to follow in the plan, should be in sorted order")


print(f"✅ PlanExecute fields: {list(PlanExecute.__annotations__)}")

In [ ]:
# ============================================================================
# PLANNER CHAIN: prompt -> structured Plan
# ============================================================================
from langchain_core.prompts import ChatPromptTemplate

planner_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """Analyze the given query or task in detail and follow these rules to make a plan:
                - Come up with a simple step by step plan (max 5 steps).
                - This plan should involve individual tasks, that if executed correctly will yield the correct answer.
                - Do not add any extra superfluous steps.
                - The result of the final step should lead to the final answer.
                - Make sure that each step has all the information needed - do not skip steps.
            """,
        ),
        ("placeholder", "{messages}"),
    ]
)

planner = planner_prompt | llm.with_structured_output(Plan)

res = planner.invoke({"messages": [("user", "who is the founder of microsoft")]})
print(f"✅ Planner test -> {res.steps}")

### 3.2 🪞 The Replanner (with Reflection)

After every executed step, the replanner sees the objective, the original plan and **every step done so far with its result**. It reflects on that and returns an `Act`: either a new `Plan` with only the steps still needed, or a `Response` - the final answer. Letting the model choose between the two types is what allows the agent to stop early, add a step it didn't foresee, or drop one that became unnecessary.

In [ ]:
# ============================================================================
# REPLANNER CHAIN: prompt -> Act (a new Plan, or the final Response)
# ============================================================================
class Response(BaseModel):
    """Response generated for the user."""

    type: Literal["Response"]
    response: str


class Act(BaseModel):
    """Action to perform."""

    action: Union[Response, Plan] = Field(
        description="Action to perform. If you want to respond to user, use Response. "
        "If you need to further use tools to get the answer, use Plan."
    )


replanner_prompt = ChatPromptTemplate.from_template(
    """For the given objective, come up with a simple step by step plan.
       This plan should involve individual tasks, that if executed correctly will yield the correct answer.
       Do not add any extra superfluous steps.
       The result of the final step should be the final answer.
       Make sure that each step has all the information needed - do not skip steps.

       Your main objective was this:
       {input}

       Your original plan was this:
       {plan}

       You have currently done the following steps (with their corresponding results):
       {past_steps}

       Update your plan accordingly as necessary.

       If no more steps are needed then analyze all the information collected so far and generate a detailed formatted answer in Markdown.
       When generating the final response in markdown, if there are special characters in the text, such as the dollar symbol,
       ensure they are escaped properly for correct rendering e.g $25.5 should become \$25.5

       Otherwise, fill out the plan.
       Only add steps to the plan that still NEED to be done.
       Do NOT return previously done steps as part of the plan."""
)

replanner = replanner_prompt | llm.with_structured_output(Act)
print("✅ Replanner chain ready")

---
## 🏗️ Part 4: The Hierarchical Plan-Execute-Replan Graph

### Key Concepts:
- **`plan_step`**: runs the planner once.
- **`make_execute_step(agent)`**: hands the **first** remaining step - plus the whole plan and the past results - to the nested research agent. It is built around the agent passed in, so Part 5 can swap in a stub.
- **`make_replan_step(replanner)`**: reflects and returns either a new plan or the response. It also enforces a **step cap in code**: after `MAX_STEPS` executed steps it stops with the findings so far, whatever the replanner wants.
- **`should_end`**: a pure router - end when a response exists, otherwise execute the next step.

In [ ]:
# ============================================================================
# NODES: plan, execute (nested agent), replan (with a step cap)
# ============================================================================
MAX_STEPS = 6  # executed steps before the replanner is overruled


async def plan_step(state: PlanExecute):
    plan = await planner.ainvoke({"messages": [("user", state["input"])]})
    print(f"📋 PLAN: {plan.steps}")
    return {"plan": plan.steps}


def make_execute_step(agent):
    async def execute_step(state: PlanExecute):
        plan, past_steps = state["plan"], state["past_steps"]
        plan_str = "\n".join(f"{i + 1}. {step}" for i, step in enumerate(plan))
        task = plan[0]
        print(f"⚙️  EXECUTE step {len(past_steps) + 1}: {task}")
        task_formatted = f"""Given the following plan:
{plan_str}

You are tasked with executing step 1, {task} only.

Here are the past steps already executed, use this information also as needed.
{past_steps}

Call tools only when you need more information.
If doing web search optimize the query for search before tool calling
otherwise refer to all the information you have to solve the step.
Do NOT call tools for analyzing existing information or compiling reports
or formatting documents.
"""
        agent_response = await agent.ainvoke({"messages": [("user", task_formatted)]})
        return {"past_steps": [(task, agent_response["messages"][-1].content)]}

    return execute_step


def make_replan_step(replanner):
    async def replan_step(state: PlanExecute):
        if len(state["past_steps"]) >= MAX_STEPS:
            print(f"🛑 REPLAN: step cap ({MAX_STEPS}) reached - stopping with the findings so far")
            findings = "\n".join(f"- {step}: {str(result)[:200]}" for step, result in state["past_steps"])
            return {"response": f"Stopped after {MAX_STEPS} steps without a final answer. Findings so far:\n{findings}"}

        output = await replanner.ainvoke(state)
        if isinstance(output.action, Response):
            print("🪞 REPLAN: enough information - writing the response")
            return {"response": output.action.response}
        print(f"🪞 REPLAN: remaining steps -> {output.action.steps}")
        return {"plan": output.action.steps}

    return replan_step


def should_end(state: PlanExecute):
    return END if state.get("response") else "react_agent"


print("✅ Nodes and router defined")

In [ ]:
# ============================================================================
# GRAPH: planner -> react_agent -> replanner -> (react_agent | END)
# ============================================================================
def build_planner_agent(agent, replanner):
    plan_graph = StateGraph(PlanExecute)
    plan_graph.add_node("planner", plan_step)
    plan_graph.add_node("react_agent", make_execute_step(agent))
    plan_graph.add_node("replanner", make_replan_step(replanner))

    plan_graph.add_edge(START, "planner")
    plan_graph.add_edge("planner", "react_agent")
    plan_graph.add_edge("react_agent", "replanner")
    plan_graph.add_conditional_edges("replanner", should_end, ["react_agent", END])
    return plan_graph.compile()


planner_agent = build_planner_agent(react_agent, replanner)

g = planner_agent.get_graph()
print("🧩 Nodes:", [n for n in g.nodes if not n.startswith("__")])
for e in g.edges:
    print(f"   {e.source:>12} ──▶ {e.target:<12} ({'conditional' if e.conditional else 'fixed'})")

### 4.1 🚀 Run the Planner Agent

A three-part question: founder, hometown, and the history of the founder and the company. Watch the replanner lines - after each step it either trims the remaining plan or decides it has enough to answer.

In [ ]:
# ============================================================================
# RUN: One multi-step question through the full hierarchy
# ============================================================================
import time

from IPython.display import Markdown, display

search_log.clear()
start = time.perf_counter()
final_state = await planner_agent.ainvoke(
    {"input": "who is the founder of microsoft, their hometown and history of founder and the company?"},
    config={"recursion_limit": 50},
)
print(f"\n📊 {len(final_state['past_steps'])} steps executed, {len(search_log)} searches, "
      f"{time.perf_counter() - start:.0f}s")
display(Markdown(final_state["response"]))

Compare the first `📋 PLAN` line with the `🪞 REPLAN` lines: the plan shrinks as steps are done, and the replanner usually stops **before** the last planned step (often a "compile the answer" step), because the results already cover it. The search count is lower than the step count whenever the research agent answered a step from the results of earlier steps - it is told to call tools only when it needs more information.

To try a harder question, change the input - for example *"Who are the key founders of Nvidia, what did they previously do and how did Nvidia become such a huge company with a huge market cap?"*.

---
## 🛑 Part 5: Proving the Cap Without an LLM

The step cap only matters when the replanner **never** decides to answer - which a live run rarely shows. Because the graph builder takes the executor agent and the replanner as arguments, stubs can reproduce that case with no model calls: the planner is still real, but the executor returns a canned result and the replanner always asks for one more step.

In [ ]:
# ============================================================================
# STUBS: An executor that returns canned text, a replanner that never answers
# ============================================================================
from langchain_core.messages import AIMessage
from langchain_core.runnables import RunnableLambda

stub_agent = RunnableLambda(lambda _: {"messages": [AIMessage(content="(canned research result)")]})
never_done_replanner = RunnableLambda(lambda _: Act(action=Plan(type="Plan", steps=["look up one more thing"])))

capped_agent = build_planner_agent(stub_agent, never_done_replanner)
capped_state = await capped_agent.ainvoke({"input": "who is the founder of microsoft"}, config={"recursion_limit": 50})

assert len(capped_state["past_steps"]) == MAX_STEPS
print(f"\n✅ Stopped after {len(capped_state['past_steps'])} steps")
print(capped_state["response"][:300])

---
## 📝 Summary

In this notebook, we learned:

### 1. A nested research agent
- A ReAct agent with a web-extraction tool becomes one component of a larger system. Truncating tool output and trimming history keep its context within a fixed budget.

### 2. Planning with a structured plan
- A prompt + `with_structured_output(Plan)` chain turns the question into a short ordered plan.

### 3. Replanning after every step
- The replanner reflects on every result so far and returns a union type - a new `Plan` or the final `Response` - so the agent can revise, extend or stop early.

### 4. A hierarchical plan-execute-replan graph
- Planner → nested ReAct executor → replanner is a hierarchical compound agent: the executor is itself an agent, but there is one overall system and one state.
- A step cap in code overrules the replanner; a prompt alone cannot guarantee termination.

### 5. Proving the cap without an LLM
- Building the graph around swappable components lets stubs show the edge case - a replanner that never finishes - deterministically and for free.

### Next Steps
- **`modules/11_reflection/notebooks/01_reflection_agents.ipynb`**: reflection as a pattern of its own - an agent that critiques and revises its own output